In [ ]:
# @title NeuG (dhaneshkk/neug hardened fork) · VisionGuard360 site graph + Agent memory — single cell
# One cell, top to bottom:
#   0. Config            · 1. Install the hardened fork wheel (vector_search + fts bundled)
#   2. Fork security probes (proves the patched build is the one running)
#   3. Lean 4 proofs of the guarantees this cell relies on (checked by the real Lean compiler)
#   4. MCC metrics + a numpy text encoder trained with Sharpness-Aware Minimization (SAM)
#   5. VisionGuard360: rig → zone → camera → equipment → permit → incident graph,
#      "incidents within 2 hops of a hot-work permit", atomic detection ingest, similarity matching
#   6. Agent memory: entities + relations, episodic embeddings, BM25 keyword recall, atomic updates,
#      hybrid recall (integer RRF) evaluated with MCC
#   7. Gemini (default) or any OpenAI-compatible /chat/completions for briefs and answers (offline fallback)
import os, sys, re, json, math, time, zlib, shutil, hashlib, tarfile, tempfile, subprocess, urllib.request, urllib.error, random
from collections import Counter, defaultdict
os.environ.setdefault("GLOG_minloglevel", "2")   # keep the C++ engine's INFO/WARNING logs out of the cell output

# ───────────────────────────── 0. Config ─────────────────────────────
CFG = dict(
    FORK_REPO="dhaneshkk/neug",
    FORK_BRANCH="security/review-fixes",
    WHEEL_RELEASE_TAG="fork-wheels",
    ALLOW_SOURCE_BUILD=False,     # True: if no wheel matches this Python, build the fork from source (slow, ~1-2 h)
    RUN_LEAN=True,                # download Lean 4 (~380 MB) and check the proofs
    LEAN_VERSION="4.21.0",
    SEED=7,
    DIM=32,                       # embedding width (FLOAT[DIM] columns)
    HASH_DIM=2048,                # hashed n-gram feature width
    SAM_RHO=0.05, LR=0.5, MOMENTUM=0.9, WEIGHT_DECAY=1e-4, EPOCHS=150,
    RRF_SCALE=1_000_000, RRF_K=60, MISS_RANK=10**9,
    TOP_K=3,
    # LLM over an OpenAI-compatible /chat/completions endpoint. Default provider: Google Gemini
    # (https://generativelanguage.googleapis.com/v1beta/openai/). "openai" = any other compatible server.
    LLM_PROVIDER="gemini",
    LLM_BASE_URL="",              # blank → provider default
    LLM_API_KEY="",               # blank → GEMINI_API_KEY / GOOGLE_API_KEY (env or Colab secret)
    LLM_MODEL="auto",             # "auto" → newest stable Gemini Flash listed by /models; or e.g. "gemini-3.6-flash"
    LLM_REASONING_EFFORT="low",   # Gemini thinking level via the OpenAI-compatible reasoning_effort field
    LLM_MAX_TOKENS=2048,          # thinking tokens count toward the output budget on Gemini 3.x
    LLM_EXTRACT_PER_EPISODE=False,  # True: also ask the LLM to extract entities for every remembered episode (~220 calls)
    WORKDIR=os.path.abspath("neug_vg360_demo"),
)
PROVIDER_DEFAULTS = {
    "gemini": dict(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", key_names=("GEMINI_API_KEY", "GOOGLE_API_KEY"),
                   model_fallback="gemini-flash-latest"),
    "openai": dict(base_url="https://api.openai.com/v1", key_names=("OPENAI_API_KEY",), model_fallback="gpt-4o-mini"),
}
def _secret(name):
    v = os.environ.get(name, "")
    if not v:
        try:
            from google.colab import userdata  # type: ignore
            v = userdata.get(name) or ""
        except Exception:
            v = ""
    return v
_prov = PROVIDER_DEFAULTS[CFG["LLM_PROVIDER"]]
CFG["LLM_BASE_URL"] = CFG["LLM_BASE_URL"] or _secret("LLM_BASE_URL") or _prov["base_url"]
CFG["LLM_API_KEY"] = CFG["LLM_API_KEY"] or next((v for v in map(_secret, _prov["key_names"]) if v), "")
if CFG["LLM_MODEL"] == "auto" and _secret("LLM_MODEL"):
    CFG["LLM_MODEL"] = _secret("LLM_MODEL")
LLM_ENABLED = bool(CFG["LLM_API_KEY"])

random.seed(CFG["SEED"])
shutil.rmtree(CFG["WORKDIR"], ignore_errors=True)
os.makedirs(CFG["WORKDIR"], exist_ok=True)
RESULTS = {}
def banner(t): print("\n" + "═" * 100 + f"\n{t}\n" + "═" * 100)

# ─────────────────────── 1. Install the hardened fork ───────────────────────
banner("1 · Install NeuG from the hardened fork " + CFG["FORK_REPO"] + "@" + CFG["FORK_BRANCH"])
def _pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

def _fork_wheel_url():
    tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    want = lambda n: n.endswith(".whl") and f"-{tag}-{tag}-" in n and "x86_64" in n
    repo, rel_tag = CFG["FORK_REPO"], CFG["WHEEL_RELEASE_TAG"]
    try:  # REST API (60 unauthenticated requests/hour per IP — shared Colab IPs can exhaust it)
        api = f"https://api.github.com/repos/{repo}/releases/tags/{rel_tag}"
        req = urllib.request.Request(api, headers={"Accept": "application/vnd.github+json", "User-Agent": "neug-colab"})
        with urllib.request.urlopen(req, timeout=60) as r:
            rel = json.load(r)
        for a in rel.get("assets", []):
            if want(a["name"]):
                return a["browser_download_url"], rel.get("target_commitish", "?")
        names = [a["name"] for a in rel.get("assets", [])]
    except urllib.error.HTTPError as e:
        print(f"GitHub API unavailable ({e.code}); reading the release asset list instead")
        req = urllib.request.Request(f"https://github.com/{repo}/releases/expanded_assets/{rel_tag}", headers={"User-Agent": "neug-colab"})
        with urllib.request.urlopen(req, timeout=60) as r:
            html = r.read().decode("utf-8", "replace")
        names = sorted(set(re.findall(r"/releases/download/[^\"']+/([^\"'/]+\.whl)", html)))
        for n in names:
            if want(n):
                return f"https://github.com/{repo}/releases/download/{rel_tag}/{n}", "?"
    raise RuntimeError(f"No {tag} x86_64 wheel in release {rel_tag}; assets: {names}")

def install_fork():
    try:
        _pip("numpy")
    except Exception:
        pass
    try:
        url, commit = _fork_wheel_url()
        print("wheel:", url.rsplit("/", 1)[-1], "| built from", commit[:12])
        _pip("--force-reinstall", "--no-deps", url)   # replace any PyPI neug with the fork build
        _pip(url)                                      # then add missing runtime deps without touching existing ones
        return "wheel"
    except Exception as e:
        print("fork wheel unavailable:", e)
        if not CFG["ALLOW_SOURCE_BUILD"]:
            raise RuntimeError("Set CFG['ALLOW_SOURCE_BUILD']=True to compile the fork from source, "
                               "or re-run once the fork-wheels release has a wheel for this Python.") from e
        env = dict(os.environ, CI_INSTALL_EXTENSIONS="vector_search;fts", BUILD_TYPE="RELEASE",
                   CMAKE_BUILD_PARALLEL_LEVEL=str(os.cpu_count() or 2))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-v",
                               f"git+https://github.com/{CFG['FORK_REPO']}@{CFG['FORK_BRANCH']}#subdirectory=tools/python_bind"], env=env)
        return "source"

INSTALL_MODE = "preinstalled (NEUG_SKIP_INSTALL)" if os.environ.get("NEUG_SKIP_INSTALL") else install_fork()
for m in [m for m in sys.modules if m == "neug" or m.startswith("neug.")]:
    del sys.modules[m]
import numpy as np
import neug
import logging
logging.getLogger("neug").setLevel(logging.WARNING)
print("neug imported from", os.path.dirname(neug.__file__), "| install:", INSTALL_MODE)

def rows(conn, q, params=None):
    res = conn.execute(q, parameters=params) if params else conn.execute(q)
    return [list(r) for r in res]

# ─────────────────────── 2. Fork security probes ───────────────────────
banner("2 · Security probes — these must be REJECTED by the hardened fork")
_probe_db = neug.Database(os.path.join(CFG["WORKDIR"], "probe.db"), mode="w")
_pc = _probe_db.connect()
def _expect_reject(q, needle):
    try:
        rows(_pc, q)
    except Exception as e:
        ok = needle.lower() in str(e).lower()
        print(("✔" if ok else "✘"), q, "→", str(e).split("Error Message:")[-1].strip()[:110])
        return ok
    print("✘", q, "→ accepted (this is NOT the hardened fork)")
    return False
SECURITY = {
    # Upstream joins the name into a path and calls remove_all(); the probe target does not exist,
    # so even an unpatched build deletes nothing — the fork rejects the name before touching disk.
    "uninstall_path_traversal_rejected": _expect_reject("UNINSTALL '../../zz_neug_probe_nonexistent'", "Invalid extension name"),
    "install_foreign_repo_rejected": _expect_reject("INSTALL gds FROM 'https://example.invalid/repo/'", "not allowed"),
}
_pc.close(); _probe_db.close()
assert all(SECURITY.values()), f"Security probes failed: {SECURITY}"
RESULTS["security"] = SECURITY

# ─────────────────────── 3. Lean 4 formal guarantees ───────────────────────
banner("3 · Lean 4 proofs (atomicity, rank-fusion monotonicity, 2-hop alert monotonicity, MCC)")
LEAN_SRC = r'''
/-! Formal guarantees for the NeuG VisionGuard360 + agent-memory notebook.
    Core Lean 4 only (no Mathlib). Each theorem mirrors a runtime check in the notebook. -/
namespace NeuGNotebook

/-! ## 1. Transaction atomicity: a write batch either fully applies or leaves state unchanged -/
structure Op (σ : Type) where
  run : σ → Option σ

def applyAll {σ : Type} : List (Op σ) → σ → Option σ
  | [], s => some s
  | o :: os, s =>
    match o.run s with
    | none => none
    | some s' => applyAll os s'

/-- Commit-or-rollback: the observable state after a transaction. -/
def txn {σ : Type} (ops : List (Op σ)) (s : σ) : σ := (applyAll ops s).getD s

theorem txn_rollback {σ : Type} (ops : List (Op σ)) (s : σ)
    (h : applyAll ops s = none) : txn ops s = s := by
  simp [txn, h]

theorem txn_commit {σ : Type} (ops : List (Op σ)) (s t : σ)
    (h : applyAll ops s = some t) : txn ops s = t := by
  simp [txn, h]

theorem applyAll_append {σ : Type} (a b : List (Op σ)) (s : σ) :
    applyAll (a ++ b) s = (applyAll a s).bind (applyAll b) := by
  induction a generalizing s with
  | nil => simp [applyAll]
  | cons o os ih =>
    simp only [List.cons_append, applyAll]
    cases o.run s with
    | none => rfl
    | some s' => exact ih s'

/-- A failure at ANY position aborts the whole batch, so the state is untouched. -/
theorem fail_anywhere_rolls_back {σ : Type} (pre post : List (Op σ)) (o : Op σ)
    (s t : σ) (hpre : applyAll pre s = some t) (hfail : o.run t = none) :
    txn (pre ++ o :: post) s = s := by
  apply txn_rollback
  rw [applyAll_append, hpre]
  simp [Option.bind, applyAll, hfail]

/-! ## 2. Hybrid retrieval fusion: integer reciprocal-rank fusion is rank-monotone -/
/-- Integer RRF contribution, exactly as computed in Python: SCALE // (K + rank). -/
def rrf (scale k r : Nat) : Nat := scale / (k + r)

theorem rrf_antitone (scale k r₁ r₂ : Nat) (hk : 0 < k) (h : r₁ ≤ r₂) :
    rrf scale k r₂ ≤ rrf scale k r₁ := by
  unfold rrf
  exact Nat.div_le_div_left (by omega) (by omega)

/-- Fused score over several ranked lists (vector, BM25, graph). -/
def fuse (scale k : Nat) : List Nat → Nat
  | [] => 0
  | r :: rs => rrf scale k r + fuse scale k rs

/-- Pointwise ≤ between two rank vectors of equal length. -/
inductive RankLe : List Nat → List Nat → Prop
  | nil : RankLe [] []
  | cons {x y : Nat} {xs ys : List Nat} : x ≤ y → RankLe xs ys → RankLe (x :: xs) (y :: ys)

/-- Improving (lowering) a document's rank in any list never lowers its fused score. -/
theorem fuse_monotone (scale k : Nat) (hk : 0 < k) :
    ∀ {xs ys : List Nat}, RankLe xs ys → fuse scale k ys ≤ fuse scale k xs
  | _, _, .nil => Nat.le_refl _
  | _, _, .cons hxy hrest =>
    Nat.add_le_add (rrf_antitone scale k _ _ hk hxy) (fuse_monotone scale k hk hrest)

/-! ## 3. Safety query monotonicity: "incidents within 2 hops of a permit" -/
/-- Undirected reachability in at most two hops (the Cypher pattern `-[*1..2]-`). -/
def Within2 {α : Type} (E : α → α → Prop) (a b : α) : Prop :=
  E a b ∨ ∃ c, E a c ∧ E c b

/-- Inserting nodes/edges (new detections, new permits) can never hide an
    incident that was already within 2 hops: alerts are monotone under inserts. -/
theorem within2_monotone {α : Type} (E E' : α → α → Prop)
    (hsub : ∀ x y, E x y → E' x y) (a b : α) (h : Within2 E a b) : Within2 E' a b := by
  rcases h with h1 | ⟨c, h1, h2⟩
  · exact Or.inl (hsub _ _ h1)
  · exact Or.inr ⟨c, hsub _ _ h1, hsub _ _ h2⟩

/-! ## 4. MCC numerator properties (the metric used for every evaluation) -/
def mccNum (tp tn fp fn : Int) : Int := tp * tn - fp * fn

theorem mcc_perfect (tp tn : Int) : mccNum tp tn 0 0 = tp * tn := by
  simp [mccNum]

/-- MCC is invariant to which class is called "positive". -/
theorem mcc_class_swap (tp tn fp fn : Int) : mccNum tp tn fp fn = mccNum tn tp fn fp := by
  simp [mccNum, Int.mul_comm]

/-- A predictor that is always wrong has a non-positive numerator. -/
theorem mcc_all_wrong (fp fn : Int) (h1 : 0 ≤ fp) (h2 : 0 ≤ fn) : mccNum 0 0 fp fn ≤ 0 := by
  simp [mccNum]
  exact Int.mul_nonneg h1 h2

end NeuGNotebook
'''
def run_lean():
    path = os.path.join(CFG["WORKDIR"], "Guarantees.lean")
    open(path, "w").write(LEAN_SRC)
    if not CFG["RUN_LEAN"]:
        print("RUN_LEAN=False — proofs written to", path, "but not checked"); return None
    v = CFG["LEAN_VERSION"]
    root = os.path.expanduser(f"~/.lean-{v}")
    lean = os.path.join(root, f"lean-{v}-linux", "bin", "lean")
    if not os.path.exists(lean):
        os.makedirs(root, exist_ok=True)
        url = f"https://github.com/leanprover/lean4/releases/download/v{v}/lean-{v}-linux.tar.zst"
        archive = os.path.join(root, "lean.tar.zst")
        print("downloading", url)
        urllib.request.urlretrieve(url, archive)
        if shutil.which("zstd"):
            subprocess.check_call(["tar", "--use-compress-program=zstd -d", "-xf", archive, "-C", root])
        else:
            _pip("zstandard"); import zstandard
            with open(archive, "rb") as fh, zstandard.ZstdDecompressor().stream_reader(fh) as rd, tarfile.open(fileobj=rd, mode="r|") as tf:
                tf.extractall(root)
        os.remove(archive)
    out = subprocess.run([lean, path], capture_output=True, text=True)
    ver = subprocess.run([lean, "--version"], capture_output=True, text=True).stdout.strip()
    ok = out.returncode == 0
    print(ver); print("✔ all theorems type-check" if ok else "✘ Lean errors:\n" + out.stdout + out.stderr)
    return ok
try:
    RESULTS["lean_ok"] = run_lean()
except Exception as e:
    print("Lean step skipped:", e); RESULTS["lean_ok"] = None
THEOREMS = ["txn_rollback", "txn_commit", "applyAll_append", "fail_anywhere_rolls_back", "rrf_antitone",
            "fuse_monotone", "within2_monotone", "mcc_perfect", "mcc_class_swap", "mcc_all_wrong"]

# ─────────────────── 4. MCC metrics + SAM-trained encoder ───────────────────
banner("4 · Metrics (MCC) and the SAM-trained text encoder")
def mcc_binary(tp, tn, fp, fn):
    num = tp * tn - fp * fn                      # Lean: mccNum
    den = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    return 0.0 if den == 0 else num / den
def confusion(y_true, y_pred):
    tp = sum(1 for t, p in zip(y_true, y_pred) if t and p); tn = sum(1 for t, p in zip(y_true, y_pred) if not t and not p)
    fp = sum(1 for t, p in zip(y_true, y_pred) if not t and p); fn = sum(1 for t, p in zip(y_true, y_pred) if t and not p)
    return tp, tn, fp, fn
def mcc_multiclass(y_true, y_pred):
    """Gorodkin's R_K statistic: MCC generalised to K classes."""
    labels = sorted(set(y_true) | set(y_pred)); idx = {l: i for i, l in enumerate(labels)}
    C = np.zeros((len(labels), len(labels)))
    for t, p in zip(y_true, y_pred): C[idx[t], idx[p]] += 1
    t_k, p_k, c, s = C.sum(1), C.sum(0), np.trace(C), C.sum()
    den = math.sqrt((s * s - (p_k * p_k).sum()) * (s * s - (t_k * t_k).sum()))
    return 0.0 if den == 0 else float((c * s - (t_k * p_k).sum()) / den)
# runtime mirrors of the Lean MCC theorems
for _ in range(500):
    tp, tn, fp, fn = (random.randint(0, 50) for _ in range(4))
    assert abs(mcc_binary(tp, tn, fp, fn) - mcc_binary(tn, tp, fn, fp)) < 1e-12        # mcc_class_swap
    assert mcc_binary(0, 0, fp, fn) <= 0                                                # mcc_all_wrong
try:
    from sklearn.metrics import matthews_corrcoef
    _yt = [random.randint(0, 3) for _ in range(300)]; _yp = [y if random.random() < .6 else random.randint(0, 3) for y in _yt]
    assert abs(matthews_corrcoef(_yt, _yp) - mcc_multiclass(_yt, _yp)) < 1e-9
    print("MCC implementation cross-checked against sklearn ✔")
except ImportError:
    pass

TOKEN_RE = re.compile(r"[a-z0-9]+")
class HashFeaturizer:
    """Signed feature hashing over words, word bigrams and char 3-grams (stable crc32, no vocabulary)."""
    def __init__(self, dim): self.dim = dim
    def tokens(self, text):
        w = TOKEN_RE.findall(text.lower().replace("_", " "))
        feats = list(w) + [f"{a}|{b}" for a, b in zip(w, w[1:])]
        for t in w:
            t = f"<{t}>"; feats += [t[i:i + 3] for i in range(len(t) - 2)]
        return feats
    def __call__(self, texts):
        X = np.zeros((len(texts), self.dim), np.float32)
        for i, t in enumerate(texts):
            for tok, c in Counter(self.tokens(t)).items():
                h = zlib.crc32(tok.encode()); X[i, h % self.dim] += (1.0 if (h >> 31) & 1 else -1.0) * (1 + math.log(c))
        n = np.linalg.norm(X, axis=1, keepdims=True); n[n == 0] = 1
        return X / n

class SAMEncoder:
    """2-layer MLP (hash features → tanh(DIM) → softmax) trained with Sharpness-Aware Minimization.
    SAM step (Foret et al. 2021): ε = ρ·g/‖g‖, then descend with ∇L(w+ε). The tanh layer is the embedding."""
    def __init__(self, in_dim, dim, n_cls, seed):
        r = np.random.default_rng(seed)
        self.p = {"W1": r.normal(0, 1 / math.sqrt(in_dim), (in_dim, dim)).astype(np.float32), "b1": np.zeros(dim, np.float32),
                  "W2": r.normal(0, 1 / math.sqrt(dim), (dim, n_cls)).astype(np.float32), "b2": np.zeros(n_cls, np.float32)}
        self.v = {k: np.zeros_like(x) for k, x in self.p.items()}
    def _loss_grad(self, p, X, Y, wd):
        H = np.tanh(X @ p["W1"] + p["b1"]); Z = H @ p["W2"] + p["b2"]
        Z = Z - Z.max(1, keepdims=True); P = np.exp(Z); P /= P.sum(1, keepdims=True)
        n = len(X); loss = -np.log(P[np.arange(n), Y] + 1e-12).mean() + 0.5 * wd * ((p["W1"] ** 2).sum() + (p["W2"] ** 2).sum())
        dZ = P; dZ[np.arange(n), Y] -= 1; dZ /= n
        g = {"W2": H.T @ dZ + wd * p["W2"], "b2": dZ.sum(0)}
        dH = (dZ @ p["W2"].T) * (1 - H ** 2)
        g["W1"] = X.T @ dH + wd * p["W1"]; g["b1"] = dH.sum(0)
        return loss, g
    def fit(self, X, Y, epochs, lr, rho, mom, wd, batch=64, seed=0):
        r = np.random.default_rng(seed); hist = []
        for ep in range(epochs):
            order = r.permutation(len(X)); ep_loss = 0.0
            for s in range(0, len(X), batch):
                b = order[s:s + batch]; Xb, Yb = X[b], Y[b]
                loss, g = self._loss_grad(self.p, Xb, Yb, wd)
                gn = math.sqrt(sum(float((x ** 2).sum()) for x in g.values())) + 1e-12
                adv = {k: self.p[k] + rho * g[k] / gn for k in self.p}            # ascend to the worst neighbour
                _, g_adv = self._loss_grad(adv, Xb, Yb, wd)                        # sharpness-aware gradient
                for k in self.p:
                    self.v[k] = mom * self.v[k] + g_adv[k]; self.p[k] -= lr * self.v[k]
                ep_loss += loss * len(b)
            hist.append(ep_loss / len(X))
        return hist
    def sharpness(self, X, Y, rho, wd):
        l0, g = self._loss_grad(self.p, X, Y, wd); gn = math.sqrt(sum(float((x ** 2).sum()) for x in g.values())) + 1e-12
        l1, _ = self._loss_grad({k: self.p[k] + rho * g[k] / gn for k in self.p}, X, Y, wd)
        return l1 - l0
    def predict(self, X): return (np.tanh(X @ self.p["W1"] + self.p["b1"]) @ self.p["W2"] + self.p["b2"]).argmax(1)
    def embed(self, X):
        H = np.tanh(X @ self.p["W1"] + self.p["b1"]); n = np.linalg.norm(H, axis=1, keepdims=True); n[n == 0] = 1
        return (H / n).astype(np.float32)

FEAT = HashFeaturizer(CFG["HASH_DIM"])
def train_encoder(texts, labels, name):
    classes = sorted(set(labels)); y = np.array([classes.index(l) for l in labels])
    r = np.random.default_rng(CFG["SEED"]); idx = r.permutation(len(texts)); cut = int(.8 * len(texts))
    tr, va = idx[:cut], idx[cut:]; X = FEAT(texts)
    enc = SAMEncoder(CFG["HASH_DIM"], CFG["DIM"], len(classes), CFG["SEED"])
    hist = enc.fit(X[tr], y[tr], CFG["EPOCHS"], CFG["LR"], CFG["SAM_RHO"], CFG["MOMENTUM"], CFG["WEIGHT_DECAY"], seed=CFG["SEED"])
    mcc_tr = mcc_multiclass(list(y[tr]), list(enc.predict(X[tr]))); mcc_va = mcc_multiclass(list(y[va]), list(enc.predict(X[va])))
    sh = enc.sharpness(X[tr], y[tr], CFG["SAM_RHO"], CFG["WEIGHT_DECAY"])
    print(f"[{name}] SAM encoder: {len(classes)} classes, loss {hist[0]:.3f}→{hist[-1]:.3f}, "
          f"train MCC {mcc_tr:.3f}, held-out MCC {mcc_va:.3f}, ρ-sharpness {sh:.4f}")
    RESULTS[f"{name}_encoder"] = dict(train_mcc=round(mcc_tr, 4), heldout_mcc=round(mcc_va, 4), sharpness=round(sh, 5))
    return enc, classes

def embed_texts(enc, texts): return enc.embed(FEAT(texts))

def rrf(rank):                                    # Lean: rrf  (SCALE / (K + rank)), integer division
    return CFG["RRF_SCALE"] // (CFG["RRF_K"] + rank)
def fuse(rank_lists, universe):
    """Integer reciprocal-rank fusion. rank_lists: list of [doc ids in rank order]. Lean: fuse / fuse_monotone."""
    pos = [{d: i + 1 for i, d in enumerate(lst)} for lst in rank_lists]
    score = {d: sum(rrf(p.get(d, CFG["MISS_RANK"])) for p in pos) for d in universe}
    return sorted(universe, key=lambda d: (-score[d], d)), score
for _ in range(1000):  # runtime mirror of fuse_monotone: improving one rank never lowers the fused score
    rs = [random.randint(1, 500) for _ in range(3)]; i = random.randrange(3); better = list(rs); better[i] = random.randint(1, rs[i])
    assert sum(map(rrf, better)) >= sum(map(rrf, rs))

def top_k_cosine(qv, ids, M, k):
    s = M @ qv; o = np.argsort(-s)[:k]; return [ids[i] for i in o]

# Extensions bundled in the fork wheel: HNSW vector search + BM25 full-text search
def load_extensions(conn):
    ok = {}
    for ext in ("vector_search", "fts"):
        try:
            rows(conn, f"LOAD {ext}"); ok[ext] = True
        except Exception as e:
            ok[ext] = False; print(f"LOAD {ext} failed → numpy/python fallback:", str(e)[:120])
    return ok

def vec_literal(v): return "CAST([" + ",".join(f"{float(x):.7f}" for x in v) + f"], 'FLOAT[{CFG['DIM']}]')"
def neug_vector_topk(conn, label, qv, k, extra_where=""):
    q = (f"MATCH (n:{label}) {extra_where} RETURN n.id, vector_distance_cosine(n.emb, {{qv}}) AS d ORDER BY d ASC LIMIT {int(k)}")
    try:
        return [r[0] for r in rows(conn, q.replace("{qv}", "$qv"), {"qv": [float(x) for x in qv]})]
    except Exception:
        return [r[0] for r in rows(conn, q.replace("{qv}", vec_literal(qv)))]

def fts_query_text(text):  # keep only plain terms (the FTS query syntax treats punctuation as operators)
    return " ".join(w for w in TOKEN_RE.findall(text.lower()) if len(w) > 2)

class BM25:
    def __init__(self, docs, k1=1.5, b=0.75):
        self.ids = list(docs); self.toks = {d: TOKEN_RE.findall(docs[d].lower()) for d in self.ids}
        self.N = len(self.ids); self.avg = sum(map(len, self.toks.values())) / max(1, self.N); self.k1, self.b = k1, b
        self.df = Counter(t for d in self.ids for t in set(self.toks[d]))
    def topk(self, q, k):
        qt = [t for t in TOKEN_RE.findall(q.lower()) if len(t) > 2]; sc = {}
        for d in self.ids:
            tf = Counter(self.toks[d]); L = len(self.toks[d]); s = 0.0
            for t in qt:
                if t in tf:
                    idf = math.log(1 + (self.N - self.df[t] + .5) / (self.df[t] + .5))
                    s += idf * tf[t] * (self.k1 + 1) / (tf[t] + self.k1 * (1 - self.b + self.b * L / self.avg))
            if s > 0: sc[d] = s
        return sorted(sc, key=lambda d: (-sc[d], d))[:k]

# ─────────────── 7 (shared). OpenAI-compatible /chat/completions (Gemini by default) ───────────────
def _llm_request(path, body=None, timeout=90, retries=4):
    url = CFG["LLM_BASE_URL"].rstrip("/") + path
    data = None if body is None else json.dumps(body).encode()
    req = urllib.request.Request(url, data=data, method="GET" if body is None else "POST",
                                 headers={"Content-Type": "application/json", "Authorization": f"Bearer {CFG['LLM_API_KEY']}"})
    for attempt in range(retries + 1):
        try:
            with urllib.request.urlopen(req, timeout=timeout) as r:
                return json.load(r)
        except urllib.error.HTTPError as e:
            detail = e.read().decode("utf-8", "replace")[:300]
            if e.code in (429, 500, 502, 503, 504) and attempt < retries:   # rate limit / overload → back off
                wait = float(e.headers.get("Retry-After") or 2 ** (attempt + 1)); time.sleep(min(wait, 60)); continue
            raise RuntimeError(f"HTTP {e.code} from {path}: {detail}") from None

_VERSION_RE = re.compile(r"gemini-(\d+(?:\.\d+)?)-flash$")
def resolve_model():
    """Pick the model once. 'auto' on Gemini → the newest stable gemini-X.Y-flash that /models lists."""
    if CFG["LLM_MODEL"] != "auto":
        return CFG["LLM_MODEL"]
    try:
        ids = [m["id"].removeprefix("models/") for m in _llm_request("/models", timeout=30).get("data", [])]
        flash = sorted(((float(m.group(1)), i) for i in ids if (m := _VERSION_RE.match(i))), reverse=True)
        if flash:
            return flash[0][1]
        print("no stable gemini-X.Y-flash in /models; using", _prov["model_fallback"])
    except Exception as e:
        print("model listing failed (", str(e)[:120], ") → using", _prov["model_fallback"])
    return _prov["model_fallback"]

def chat(messages, max_tokens=None, json_mode=False):
    """POST {base}/chat/completions. Returns the reply text, or None when no key is set or the call fails.
    No temperature/top_p: Gemini 3.6+ ignores them and newer generations reject them with HTTP 400."""
    if not LLM_ENABLED:
        return None
    body = {"model": CFG["LLM_MODEL"], "messages": messages, "max_tokens": max_tokens or CFG["LLM_MAX_TOKENS"]}
    if CFG["LLM_REASONING_EFFORT"]:
        body["reasoning_effort"] = CFG["LLM_REASONING_EFFORT"]
    if json_mode:
        body["response_format"] = {"type": "json_object"}
    try:
        try:
            out = _llm_request("/chat/completions", body)
        except RuntimeError as e:  # older/other models may not accept reasoning_effort or response_format
            if "HTTP 400" not in str(e):
                raise
            body.pop("reasoning_effort", None); body.pop("response_format", None)
            out = _llm_request("/chat/completions", body)
        msg = out["choices"][0]["message"]
        text = msg.get("content") or ""
        if isinstance(text, list):  # some servers return content parts
            text = "".join(p.get("text", "") for p in text if isinstance(p, dict))
        if not text.strip():
            print("LLM returned no text (finish_reason=%s) — raise CFG['LLM_MAX_TOKENS']" % out["choices"][0].get("finish_reason"))
            return None
        return text.strip()
    except Exception as e:
        print("chat/completions unavailable, using offline fallback:", str(e)[:160]); return None

def parse_json_reply(text):
    """Tolerate ```json fences and prose around the object."""
    if not text:
        return None
    t = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip())
    try:
        return json.loads(t)
    except Exception:
        m = re.search(r"\{.*\}", t, re.S)
        try:
            return json.loads(m.group(0)) if m else None
        except Exception:
            return None

banner("7 · LLM endpoint")
if LLM_ENABLED:
    CFG["LLM_MODEL"] = resolve_model()
    pong = chat([{"role": "user", "content": "Reply with the single word: ready"}], max_tokens=256)
    print(f"provider {CFG['LLM_PROVIDER']} | model {CFG['LLM_MODEL']} | {CFG['LLM_BASE_URL']}chat/completions | smoke test: {pong!r}")
    LLM_ENABLED = pong is not None
else:
    print(f"No API key for provider '{CFG['LLM_PROVIDER']}' (set {' or '.join(_prov['key_names'])} as a Colab secret) → offline fallbacks")
RESULTS["llm"] = {"provider": CFG["LLM_PROVIDER"], "model": CFG["LLM_MODEL"] if LLM_ENABLED else None, "enabled": LLM_ENABLED}

# ═══════════════════════ 5. VisionGuard360 site graph ═══════════════════════
banner("5 · VisionGuard360 — rig → zone → camera → equipment → permit → incident")
rng = random.Random(CFG["SEED"])
CATS = ["gas_release", "hot_work_fire", "dropped_object", "ppe_violation", "confined_space", "spill"]
HIGH_RISK_NEAR_HOT_WORK = {"gas_release", "hot_work_fire"}
ZONES = [("Z-DF", "Drill floor", "Zone 1"), ("Z-PD", "Pipe deck", "Zone 2"), ("Z-MP", "Mud pit room", "Zone 1"),
         ("Z-PR", "Process & separator skid", "Zone 1"), ("Z-UT", "Utilities & generators", "Zone 2"), ("Z-AC", "Accommodation", "Unclassified")]
EQUIP = [("EQ-TD", "Top drive", "Z-DF"), ("EQ-DW", "Drawworks", "Z-DF"), ("EQ-PC", "Pipe crane", "Z-PD"), ("EQ-RK", "Pipe rack", "Z-PD"),
         ("EQ-MP1", "Mud pump 1", "Z-MP"), ("EQ-SS", "Shale shaker", "Z-MP"), ("EQ-MT", "Mud tank", "Z-MP"),
         ("EQ-SEP", "Test separator", "Z-PR"), ("EQ-FL", "Flare line", "Z-PR"), ("EQ-GEN", "Diesel generator", "Z-UT"),
         ("EQ-DT", "Diesel day tank", "Z-UT"), ("EQ-GW", "Galley", "Z-AC")]
EQ_ZONE = {e: z for e, _, z in EQUIP}; EQ_NAME = {e: n for e, n, _ in EQUIP}
CAMS = [("CAM-01", "Z-DF"), ("CAM-02", "Z-DF"), ("CAM-03", "Z-PD"), ("CAM-04", "Z-MP"), ("CAM-05", "Z-PR"), ("CAM-06", "Z-PR"),
        ("CAM-07", "Z-UT"), ("CAM-08", "Z-AC")]
ZONE_CAMS = defaultdict(list)
for c, z in CAMS: ZONE_CAMS[z].append(c)
PERMITS = [("PTW-HW-0412", "HOT_WORK", "ACTIVE", ["EQ-SEP", "EQ-FL"]), ("PTW-HW-0415", "HOT_WORK", "ACTIVE", ["EQ-PC"]),
           ("PTW-CS-0398", "CONFINED_SPACE", "ACTIVE", ["EQ-MT"]), ("PTW-HW-0377", "HOT_WORK", "CLOSED", ["EQ-GEN"]),
           ("PTW-EL-0420", "ELECTRICAL_ISOLATION", "ACTIVE", ["EQ-MP1"])]
# which hazards each equipment tends to produce (drives realistic graph↔text correlation)
EQ_CAT_W = {"EQ-TD": {"dropped_object": 4, "ppe_violation": 2}, "EQ-DW": {"dropped_object": 3, "ppe_violation": 2},
            "EQ-PC": {"dropped_object": 4, "hot_work_fire": 1}, "EQ-RK": {"dropped_object": 3, "ppe_violation": 2},
            "EQ-MP1": {"spill": 3, "ppe_violation": 1}, "EQ-SS": {"spill": 3, "gas_release": 1}, "EQ-MT": {"confined_space": 4, "gas_release": 2},
            "EQ-SEP": {"gas_release": 4, "hot_work_fire": 2}, "EQ-FL": {"gas_release": 3, "hot_work_fire": 3},
            "EQ-GEN": {"hot_work_fire": 2, "spill": 2}, "EQ-DT": {"spill": 4, "hot_work_fire": 1}, "EQ-GW": {"hot_work_fire": 2, "ppe_violation": 1}}
REPORT_T = {
    "gas_release": ["H2S alarm at {eq}; personnel mustered upwind, gas reading {n} ppm", "hydrocarbon release detected near {eq} flange, LEL at {n} percent",
                    "fixed gas detector tripped around {eq}, suspected leaking valve packing", "gas cloud observed at {eq}, ESD initiated and area isolated"],
    "hot_work_fire": ["sparks from grinding at {eq} landed on oily rags, small fire extinguished", "welding near {eq} without fire watch, smouldering insulation",
                      "hot work at {eq} ignited residue, fire blanket used", "flash fire at {eq} during cutting, permit conditions not met"],
    "dropped_object": ["spanner fell from height near {eq}, exclusion zone not barriered", "dropped object from derrick above {eq}, no tether on tool",
                       "unsecured bolt dropped {n} metres beside {eq}", "lifting sling slipped at {eq}, load fell to deck"],
    "ppe_violation": ["worker without hard hat observed at {eq}", "missing safety harness while working at height on {eq}",
                      "no gloves or eye protection during task at {eq}", "crew member entered red zone at {eq} without helmet"],
    "confined_space": ["entry into {eq} without gas test or standby man", "worker inside {eq} with expired confined space permit",
                       "low oxygen reading {n} percent inside {eq}, entrant evacuated", "rescue plan missing for entry into {eq}"],
    "spill": ["drilling mud overflow at {eq}, approx {n} litres on deck", "diesel leak from {eq} hose, drip tray overflowing",
              "oil sheen near {eq}, spill kit deployed", "hydraulic fluid spill at {eq} from ruptured line"]}
DETECT_T = {  # camera-analytics phrasing; variants 0-1 may be labelled for training, variant 2 is never seen in training
    "gas_release": ["vapour plume visible at {eq}, thermal anomaly cold spot", "optical gas imaging haze detected near {eq}",
                    "shimmering cloud drifting from {eq}, people moving away upwind"],
    "hot_work_fire": ["flame and smoke detected near {eq}, conf {c}", "spark shower detected by thermal camera at {eq}",
                      "bright glow and hot spot on thermal feed at {eq}, smoke rising"],
    "dropped_object": ["falling object trajectory detected from height above {eq}", "object impact on deck detected beside {eq}",
                       "item tumbling from the derrick lands next to {eq}"],
    "ppe_violation": ["person detected no helmet near {eq}, conf {c}", "person at height without harness detected at {eq}",
                      "bare head worker leaning over {eq} railing"],
    "confined_space": ["person entering manway of {eq} with no attendant detected", "person count inside {eq} hatch exceeds permit",
                       "worker disappears into {eq} opening, nobody standing by"],
    "spill": ["liquid pooling detected on deck at {eq}", "spreading dark fluid patch detected next to {eq}",
              "glossy wet patch growing around {eq} base"]}
def wpick(d): ks = list(d); return rng.choices(ks, [d[k] for k in ks])[0]
CONTEXT = ["during night shift", "reported by drill crew", "near the fire watch station", "after crane lift", "spill kit nearby",
           "harness inspection overdue", "gas test logged earlier", "while hot work permit was open", "toolbox talk held",
           "area barriered afterwards", "flagged by HSE walk", "deck wet after rain", "high winds on deck", "shift handover"]
def noisy(txt):
    extra = rng.sample(CONTEXT, rng.randint(1, 3))
    return (txt + ", " + ", ".join(extra)) if rng.random() < .8 else (", ".join(extra) + ": " + txt)

incidents = []
for i in range(360):
    eq = rng.choice(EQUIP)[0]; cat = wpick(EQ_CAT_W[eq]) if rng.random() < .85 else rng.choice(CATS)
    txt = noisy(rng.choice(REPORT_T[cat]).format(eq=EQ_NAME[eq].lower(), n=rng.randint(5, 80)))
    incidents.append(dict(id=f"INC-{i:04d}", ts=1_700_000_000 + i * 86_400, category=cat, severity=rng.randint(1, 5), txt=txt,
                          eq=eq if rng.random() < .8 else None, zone=EQ_ZONE[eq], cam=rng.choice(ZONE_CAMS[EQ_ZONE[eq]])))
# ── New detections (held-out, camera-analytics vocabulary) → similarity matching
detections = []
for j in range(240):
    eq = rng.choice(EQUIP)[0]; cat = wpick(EQ_CAT_W[eq]) if rng.random() < .85 else rng.choice(CATS)
    detections.append(dict(id=f"DET-{j:04d}", ts=1_731_000_000 + j * 600, true_cat=cat, severity=rng.randint(2, 5), eq=eq, zone=EQ_ZONE[eq],
                           cam=rng.choice(ZONE_CAMS[EQ_ZONE[eq]]),
                           txt=noisy(DETECT_T[cat][rng.randint(0, 1) if j < 90 else 2].format(eq=EQ_NAME[eq].lower(), c=f"0.{rng.randint(55, 99)}"))))
# 90 labelled camera detections (operator-confirmed, phrasings 0-1) join the reports in training;
# the 150 evaluation detections use phrasing 2, which the encoder never saw
train_dets, eval_dets = detections[:90], detections[90:]
vg_enc, vg_classes = train_encoder([x["txt"] for x in incidents] + [d["txt"] for d in train_dets],
                                   [x["category"] for x in incidents] + [d["true_cat"] for d in train_dets], "vg360")
for x, e in zip(incidents, embed_texts(vg_enc, [x["txt"] for x in incidents])): x["emb"] = e

vg_db = neug.Database(os.path.join(CFG["WORKDIR"], "vg360.db"), mode="w"); vg = vg_db.connect()
VG_EXT = load_extensions(vg)
D = CFG["DIM"]
for ddl in ["CREATE NODE TABLE Rig(id STRING, name STRING, PRIMARY KEY(id))",
            "CREATE NODE TABLE Zone(id STRING, name STRING, hazard_class STRING, PRIMARY KEY(id))",
            "CREATE NODE TABLE Camera(id STRING, PRIMARY KEY(id))",
            "CREATE NODE TABLE Equipment(id STRING, name STRING, PRIMARY KEY(id))",
            "CREATE NODE TABLE Permit(id STRING, kind STRING, status STRING, PRIMARY KEY(id))",
            f"CREATE NODE TABLE Incident(id STRING, ts INT64, category STRING, severity INT64, source STRING, txt STRING, emb FLOAT[{D}], PRIMARY KEY(id))",
            "CREATE REL TABLE HAS_ZONE(FROM Rig TO Zone)", "CREATE REL TABLE MONITORS(FROM Camera TO Zone)",
            "CREATE REL TABLE LOCATED_IN(FROM Equipment TO Zone)", "CREATE REL TABLE COVERS(FROM Permit TO Equipment)",
            "CREATE REL TABLE OCCURRED_IN(FROM Incident TO Zone)", "CREATE REL TABLE INVOLVES(FROM Incident TO Equipment)",
            "CREATE REL TABLE DETECTED_BY(FROM Incident TO Camera)"]:
    rows(vg, ddl)

t0 = time.time()
vg.begin_transaction()  # bulk seed as ONE atomic transaction
rows(vg, "CREATE (:Rig {id:'RIG-07', name:'Jack-up RIG-07'})")
for z, n, h in ZONES:
    rows(vg, "CREATE (:Zone {id:$id, name:$n, hazard_class:$h})", {"id": z, "n": n, "h": h})
    rows(vg, "MATCH (r:Rig {id:'RIG-07'}),(z:Zone {id:$z}) CREATE (r)-[:HAS_ZONE]->(z)", {"z": z})
for c, z in CAMS:
    rows(vg, "CREATE (:Camera {id:$id})", {"id": c}); rows(vg, "MATCH (c:Camera {id:$c}),(z:Zone {id:$z}) CREATE (c)-[:MONITORS]->(z)", {"c": c, "z": z})
for e, n, z in EQUIP:
    rows(vg, "CREATE (:Equipment {id:$id, name:$n})", {"id": e, "n": n}); rows(vg, "MATCH (e:Equipment {id:$e}),(z:Zone {id:$z}) CREATE (e)-[:LOCATED_IN]->(z)", {"e": e, "z": z})
for p, k, s, eqs in PERMITS:
    rows(vg, "CREATE (:Permit {id:$id, kind:$k, status:$s})", {"id": p, "k": k, "s": s})
    for e in eqs: rows(vg, "MATCH (p:Permit {id:$p}),(e:Equipment {id:$e}) CREATE (p)-[:COVERS]->(e)", {"p": p, "e": e})
for x in incidents:
    rows(vg, "CREATE (:Incident {id:$id, ts:$ts, category:$c, severity:$s, source:'report', txt:$t, emb:$emb})",
         {"id": x["id"], "ts": x["ts"], "c": x["category"], "s": x["severity"], "t": x["txt"], "emb": [float(v) for v in x["emb"]]})
    rows(vg, "MATCH (i:Incident {id:$i}),(z:Zone {id:$z}) CREATE (i)-[:OCCURRED_IN]->(z)", {"i": x["id"], "z": x["zone"]})
    rows(vg, "MATCH (i:Incident {id:$i}),(c:Camera {id:$c}) CREATE (i)-[:DETECTED_BY]->(c)", {"i": x["id"], "c": x["cam"]})
    if x["eq"]: rows(vg, "MATCH (i:Incident {id:$i}),(e:Equipment {id:$e}) CREATE (i)-[:INVOLVES]->(e)", {"i": x["id"], "e": x["eq"]})
vg.commit()
if VG_EXT["vector_search"]:
    rows(vg, "CREATE INDEX incident_hnsw IF NOT EXISTS ON Incident USING HNSW (emb) WITH (metric='cosine', m=16, ef_construction=200)")
print(f"seeded {len(incidents)} incidents + topology in {time.time() - t0:.1f}s | extensions: {VG_EXT}")
print("node counts:", {l: rows(vg, f"MATCH (n:{l}) RETURN count(n)")[0][0] for l in ["Zone", "Camera", "Equipment", "Permit", "Incident"]})

# ── Q1: incidents within 2 hops of a hot-work permit (verified against an independent Python BFS)
Q_2HOP = ("MATCH (p:Permit {id:$pid})-[*1..2]-(i:Incident) "
          "RETURN DISTINCT i.id, i.category, i.severity, i.txt ORDER BY i.severity DESC, i.id")
def two_hop_reference(pid, incs):
    covered = {e for p, _, _, eqs in PERMITS if p == pid for e in eqs}
    return {x["id"] for x in incs if x["eq"] in covered}
pid = "PTW-HW-0412"
q1 = rows(vg, Q_2HOP, {"pid": pid}); q1_ids = {r[0] for r in q1}
assert q1_ids == two_hop_reference(pid, incidents), "2-hop query disagrees with reference BFS"
print(f"\nQ1 · incidents within 2 hops of {pid} (hot work on {', '.join(EQ_NAME[e] for e in PERMITS[0][3])}): {len(q1)} "
      f"— matches reference BFS ✔")
for r in q1[:5]: print(f"   {r[0]}  sev {r[2]}  {r[1]:<15} {r[3]}")
print("   by category:", dict(Counter(r[1] for r in q1)))

# ── Q2: zone risk board — active hot work vs. incident history per zone
q2 = rows(vg, "MATCH (z:Zone)<-[:OCCURRED_IN]-(i:Incident) RETURN z.id, z.name, z.hazard_class, count(i) AS n, "
              "sum(CASE WHEN i.severity >= 4 THEN 1 ELSE 0 END) AS sev4 ORDER BY sev4 DESC")
hw = rows(vg, "MATCH (p:Permit {kind:'HOT_WORK', status:'ACTIVE'})-[:COVERS]->(e:Equipment)-[:LOCATED_IN]->(z:Zone) RETURN z.id, collect(p.id)")
hw = {z: sorted(set(ps)) for z, ps in hw}
print("\nQ2 · zone risk board")
for zid, zn, hc, n, s4 in q2: print(f"   {zn:<26} {hc:<13} incidents {n:>3}  sev≥4 {s4:>3}  active hot work: {', '.join(hw.get(zid, [])) or '—'}")

# ── Atomic ingest of a new camera detection (all edges or nothing)
class InjectedFailure(Exception): pass
def ingest_detection(det, fail_after=None):
    """One explicit transaction: Incident node + OCCURRED_IN + DETECTED_BY (+ INVOLVES). fail_after=k injects a crash after k writes."""
    steps = [("CREATE (:Incident {id:$id, ts:$ts, category:$c, severity:$s, source:'camera', txt:$t, emb:$emb})",
              {"id": det["id"], "ts": det["ts"], "c": det["pred"], "s": det["severity"], "t": det["txt"], "emb": [float(v) for v in det["emb"]]}),
             ("MATCH (i:Incident {id:$i}),(z:Zone {id:$z}) CREATE (i)-[:OCCURRED_IN]->(z)", {"i": det["id"], "z": det["zone"]}),
             ("MATCH (i:Incident {id:$i}),(c:Camera {id:$c}) CREATE (i)-[:DETECTED_BY]->(c)", {"i": det["id"], "c": det["cam"]})]
    if det.get("eq"): steps.append(("MATCH (i:Incident {id:$i}),(e:Equipment {id:$e}) CREATE (i)-[:INVOLVES]->(e)", {"i": det["id"], "e": det["eq"]}))
    vg.begin_transaction()
    try:
        if not rows(vg, "MATCH (c:Camera {id:$c})-[:MONITORS]->(z:Zone {id:$z}) RETURN c.id", {"c": det["cam"], "z": det["zone"]}):
            raise ValueError(f"camera {det['cam']} does not monitor {det['zone']}")
        for k, (q, p) in enumerate(steps):
            if fail_after is not None and k == fail_after: raise InjectedFailure(f"simulated crash after {k} writes")
            rows(vg, q, p)
        vg.commit()
    except Exception:
        vg.rollback(); raise

def graph_state():
    return (rows(vg, "MATCH (i:Incident) RETURN count(i)")[0][0], rows(vg, "MATCH ()-[r]->() RETURN count(r)")[0][0])

hist_ids = [x["id"] for x in incidents]; hist_cat = {x["id"]: x["category"] for x in incidents}
hist_M = np.stack([x["emb"] for x in incidents]); hist_raw = FEAT([x["txt"] for x in incidents])
det_emb = embed_texts(vg_enc, [d["txt"] for d in eval_dets]); det_raw = FEAT([d["txt"] for d in eval_dets])
inc_by_eq = defaultdict(list); inc_by_zone = defaultdict(list)
for x in sorted(incidents, key=lambda x: (-x["severity"], -x["ts"])):
    if x["eq"]: inc_by_eq[x["eq"]].append(x["id"])
    inc_by_zone[x["zone"]].append(x["id"])
def graph_ranked(det):  # structural proximity: same equipment first, then same zone
    seen, out = set(), []
    for i in inc_by_eq[det["eq"]] + inc_by_zone[det["zone"]]:
        if i not in seen: seen.add(i); out.append(i)
    return out
def vote(ids): return Counter(hist_cat[i] for i in ids).most_common(1)[0][0]
K = 5
preds = {"hash features (no training)": [], "SAM encoder": [], "SAM encoder + graph RRF": []}
agree = []
for d, e, raw in zip(eval_dets, det_emb, det_raw):
    preds["hash features (no training)"].append(vote(top_k_cosine(raw, hist_ids, hist_raw, K)))
    vec = top_k_cosine(e, hist_ids, hist_M, 50)
    preds["SAM encoder"].append(vote(vec[:K]))
    fused, _ = fuse([vec, graph_ranked(d)[:50]], set(vec) | set(graph_ranked(d)[:50]))
    preds["SAM encoder + graph RRF"].append(vote(fused[:K]))
    if VG_EXT["vector_search"] and len(agree) < 30:
        agree.append(len(set(neug_vector_topk(vg, "Incident", e, K, "WHERE n.source = 'report'")) & set(vec[:K])) / K)
truth = [d["true_cat"] for d in eval_dets]
print(f"\nDetection → past-incident matching (top-5 vote), {len(eval_dets)} unseen camera detections")
RESULTS["vg360_matching_mcc"] = {}
for name, p in preds.items():
    m = mcc_multiclass(truth, p); RESULTS["vg360_matching_mcc"][name] = round(m, 4); print(f"   {name:<30} multiclass MCC {m:.3f}")
if agree: print(f"   NeuG HNSW vs exact numpy top-5 overlap: {np.mean(agree):.2f}"); RESULTS["vg360_hnsw_overlap"] = round(float(np.mean(agree)), 3)

# STOP-WORK alert: high-risk class AND the equipment is covered by an ACTIVE hot-work permit (2-hop pattern)
ACTIVE_HW_EQ = {e for p, k, s, eqs in PERMITS if k == "HOT_WORK" and s == "ACTIVE" for e in eqs}
def alert(cat, eq): return cat in HIGH_RISK_NEAR_HOT_WORK and eq in ACTIVE_HW_EQ
y_al = [alert(d["true_cat"], d["eq"]) for d in eval_dets]; p_al = [alert(c, d["eq"]) for c, d in zip(preds["SAM encoder + graph RRF"], eval_dets)]
cm = confusion(y_al, p_al); m_al = mcc_binary(*cm)
print(f"   STOP-WORK alert (high-risk ∧ active hot-work 2-hop): binary MCC {m_al:.3f}  (tp,tn,fp,fn)={cm}")
RESULTS["vg360_alert_mcc"] = round(m_al, 4)

# Ingest one detection atomically; prove rollback; prove the 2-hop alert set only grows (Lean: within2_monotone)
PID_EQ = {e for p, _, _, eqs in PERMITS if p == pid for e in eqs}
det = next(d for d, c in zip(eval_dets, preds["SAM encoder + graph RRF"]) if d["eq"] in PID_EQ and c in HIGH_RISK_NEAR_HOT_WORK)
det["pred"] = preds["SAM encoder + graph RRF"][eval_dets.index(det)]; det["emb"] = det_emb[eval_dets.index(det)]
before_state = graph_state(); before_2hop = {r[0] for r in rows(vg, Q_2HOP, {"pid": pid})}
for k in range(4):
    try: ingest_detection(det, fail_after=k)
    except InjectedFailure: pass
    assert graph_state() == before_state, f"partial write survived a failure after {k} writes"
try: ingest_detection(dict(det, cam="CAM-08"))  # camera that does not watch this zone → validation failure → rollback
except ValueError: pass
assert graph_state() == before_state
print(f"\nAtomicity: injected failures after 0..3 writes and a validation failure → graph unchanged {before_state} ✔ (Lean: fail_anywhere_rolls_back)")
ingest_detection(det)
after_state = graph_state(); after_2hop = {r[0] for r in rows(vg, Q_2HOP, {"pid": pid})}
assert before_2hop <= after_2hop and det["id"] in after_2hop
print(f"Committed {det['id']}: {before_state} → {after_state}; 2-hop set {len(before_2hop)} → {len(after_2hop)}, old ⊆ new ✔ (Lean: within2_monotone)")
similar = top_k_cosine(det["emb"], hist_ids, hist_M, 3)
ctx = rows(vg, "MATCH (i:Incident) WHERE i.id IN $ids RETURN i.id, i.category, i.severity, i.txt", {"ids": similar})
brief_prompt = [{"role": "system", "content": "You are an offshore HSE assistant. Write a 4-line STOP-WORK brief: hazard, location, permit, immediate action."},
                {"role": "user", "content": json.dumps({"detection": det["txt"], "camera": det["cam"], "equipment": EQ_NAME[det["eq"]],
                                                         "zone": dict((z, n) for z, n, _ in ZONES)[det["zone"]],
                                                         "active_permits": [p for p, k, s, eqs in PERMITS if det["eq"] in eqs and s == "ACTIVE"],
                                                         "similar_past_incidents": ctx})}]
brief = chat(brief_prompt) or (
    f"HAZARD: {det['pred'].replace('_', ' ')} — {det['txt']}\nLOCATION: {EQ_NAME[det['eq']]} ({det['zone']}), seen by {det['cam']}\n"
    f"PERMIT: {', '.join(p for p, k, s, eqs in PERMITS if det['eq'] in eqs and s == 'ACTIVE')} — suspend hot work now\n"
    f"PRECEDENT: {'; '.join(f'{r[0]} sev{r[2]} {r[3]}' for r in ctx[:2])}")
print("\nSTOP-WORK brief" + (f" ({CFG['LLM_MODEL']} via /chat/completions)" if LLM_ENABLED else " (offline template)") + ":\n" + brief)

# ═══════════════════════════ 6. Agent memory ═══════════════════════════
banner("6 · Agent memory — entities + relations, episodic embeddings, keyword recall, atomic updates")
PEOPLE = {"faisal": ("Faisal", "person"), "priya": ("Priya", "person"), "omar": ("Omar", "person"), "lina": ("Lina", "person")}
ORGS = {"rimal": ("Rimal Energy", "client"), "najm": ("Najm Offshore", "client")}
SITES = {"rig07": ("RIG-07", "site"), "rig11": ("RIG-11", "site")}
THINGS = {"ppe_model": ("PPE detector v3", "model"), "ogi_cam": ("OGI camera", "device"), "h2s_gw": ("H2S gateway", "device")}
ENT = {**PEOPLE, **ORGS, **SITES, **THINGS}
NAME2ID = {v[0].lower(): k for k, v in ENT.items()}
TOPICS = {
    "client_pref": [("{org} wants alert messages in Arabic and English for {site}", ["org", "site"]),
                    ("{org} asked that STOP-WORK alerts escalate to the OIM within 60 seconds", ["org"]),
                    ("{org} prefers a weekly PDF safety digest every Sunday morning", ["org"])],
    "personnel": [("{person} is the night shift supervisor on {site}", ["person", "site"]),
                  ("{person} is the HSE manager for {org}", ["person", "org"]),
                  ("{person} moved to the day shift rotation on {site}", ["person", "site"])],
    "model_ops": [("{thing} false positives on {site} rose after the deck lighting change", ["thing", "site"]),
                  ("retrained {thing} with night images from {site}, precision improved", ["thing", "site"]),
                  ("{person} owns the retraining pipeline for {thing}", ["person", "thing"])],
    "hardware": [("{thing} on {site} needs a new mounting bracket near the flare boom", ["thing", "site"]),
                 ("firmware update for {thing} scheduled on {site} next Tuesday", ["thing", "site"]),
                 ("{thing} lost network link on {site} during the storm", ["thing", "site"])],
    "billing": [("{org} invoice for {site} cameras is due at month end", ["org", "site"]),
                ("{org} requested a quote for adding four cameras to {site}", ["org", "site"])]}
def fill(t, slots):
    pick = {"person": rng.choice(list(PEOPLE)), "org": rng.choice(list(ORGS)), "site": rng.choice(list(SITES)), "thing": rng.choice(list(THINGS))}
    txt = t.format(**{s: ENT[pick[s]][0] for s in ("person", "org", "site", "thing")})
    return txt, [pick[s] for s in slots]
episodes = []
for i in range(220):
    topic = rng.choice(list(TOPICS)); t, slots = rng.choice(TOPICS[topic]); txt, ents = fill(t, slots)
    episodes.append(dict(id=f"EP-{i:04d}", ts=1_731_000_000 + i * 3600, session=f"S{i // 20:02d}", topic=topic, txt=txt, ents=ents))
REL_PATTERNS = [(re.compile(r"^(\w+) is the (night shift supervisor|HSE manager)(?: on| for) (.+)$", re.I), "role"),
                (re.compile(r"^(\w+) moved to the (day shift) rotation on (.+)$", re.I), "role"),
                (re.compile(r"^(\w+) owns the (retraining pipeline) for (.+)$", re.I), "owns")]
def extract(text):
    """Deterministic gazetteer + pattern extractor. With an LLM endpoint configured, /chat/completions JSON extraction is merged in."""
    low = text.lower(); ents = sorted({NAME2ID[n] for n in NAME2ID if n in low}); rels = []
    for rx, kind in REL_PATTERNS:
        m = rx.match(text)
        if m and m.group(1).lower() in NAME2ID and m.group(3).lower() in NAME2ID:
            rels.append((NAME2ID[m.group(1).lower()], f"{kind}:{m.group(2).lower()}", NAME2ID[m.group(3).lower()]))
    return ents, rels

mem_enc, mem_topics = train_encoder([e["txt"] for e in episodes], [e["topic"] for e in episodes], "memory")
mem_db_path = os.path.join(CFG["WORKDIR"], "agent_memory.db")
mem_db = neug.Database(mem_db_path, mode="w"); mem = mem_db.connect()
MEM_EXT = load_extensions(mem)
for ddl in ["CREATE NODE TABLE Entity(id STRING, name STRING, kind STRING, PRIMARY KEY(id))",
            f"CREATE NODE TABLE Episode(id STRING, ts INT64, session STRING, txt STRING, emb FLOAT[{D}], PRIMARY KEY(id))",
            "CREATE REL TABLE MENTIONS(FROM Episode TO Entity)",
            "CREATE REL TABLE RELATED(FROM Entity TO Entity, rel STRING, ts INT64)"]:
    rows(mem, ddl)

def remember(ep, fail_after=None, plan_only=False):
    """Atomic memory write: Episode + embedding + entity upserts + MENTIONS + RELATED facts in ONE transaction.
    A newer fact with the same (subject, relation kind) supersedes the old edge inside the same transaction."""
    ents, rels = extract(ep["txt"])
    if LLM_ENABLED and CFG["LLM_EXTRACT_PER_EPISODE"] and fail_after is None and not plan_only:
        out = parse_json_reply(chat([{"role": "system", "content": "Extract named entities. Reply with JSON {\"entities\": [names]} only."},
                                     {"role": "user", "content": ep["txt"]}], max_tokens=512, json_mode=True))
        if isinstance(out, dict) and isinstance(out.get("entities"), list):
            ents = sorted(set(ents) | {NAME2ID[n.lower()] for n in out["entities"] if isinstance(n, str) and n.lower() in NAME2ID})
    emb = embed_texts(mem_enc, [ep["txt"]])[0]
    ops = [("CREATE (:Episode {id:$id, ts:$ts, session:$s, txt:$t, emb:$emb})",
            {"id": ep["id"], "ts": ep["ts"], "s": ep["session"], "t": ep["txt"], "emb": [float(v) for v in emb]})]
    for e in ents:
        ops.append(("MERGE (n:Entity {id:$id}) ON CREATE SET n.name=$n, n.kind=$k", {"id": e, "n": ENT[e][0], "k": ENT[e][1]}))
        ops.append(("MATCH (p:Episode {id:$p}),(n:Entity {id:$e}) CREATE (p)-[:MENTIONS]->(n)", {"p": ep["id"], "e": e}))
    for s, r, o in rels:
        kind = r.split(":")[0]
        ops.append(("MATCH (a:Entity {id:$s})-[x:RELATED]->(b:Entity {id:$o}) WHERE x.rel STARTS WITH $kind DELETE x", {"s": s, "o": o, "kind": kind + ":"}))
        ops.append(("MATCH (a:Entity {id:$s}),(b:Entity {id:$o}) CREATE (a)-[:RELATED {rel:$r, ts:$ts}]->(b)", {"s": s, "o": o, "r": r, "ts": ep["ts"]}))
    if plan_only:
        return len(ops)
    mem.begin_transaction()
    try:
        for k, (q, p) in enumerate(ops):
            if fail_after is not None and k == fail_after: raise InjectedFailure(f"crash after {k} writes")
            rows(mem, q, p)
        mem.commit()
    except Exception:
        mem.rollback(); raise
    return len(ops)

def mem_state():
    return tuple(rows(mem, q)[0][0] for q in ["MATCH (e:Episode) RETURN count(e)", "MATCH (n:Entity) RETURN count(n)", "MATCH ()-[r]->() RETURN count(r)"])

t0 = time.time()
for ep in episodes[:-1]: remember(ep)
if MEM_EXT["vector_search"]:
    rows(mem, "CREATE INDEX episode_hnsw IF NOT EXISTS ON Episode USING HNSW (emb) WITH (metric='cosine', m=16, ef_construction=200)")
if MEM_EXT["fts"]:
    rows(mem, "CREATE INDEX episode_fts IF NOT EXISTS ON Episode USING FTS (txt) WITH (stopwords='english')")
print(f"remembered {len(episodes) - 1} episodes in {time.time() - t0:.1f}s | state (episodes, entities, edges) = {mem_state()} | extensions: {MEM_EXT}")

# Atomicity under injected crashes (indexes included: HNSW + FTS are maintained inside the same transaction)
last = episodes[-1]; s0 = mem_state(); n_ops = remember(last, plan_only=True)
for k in range(1, n_ops):
    try: remember(last, fail_after=k)
    except InjectedFailure: pass
    assert mem_state() == s0, f"partial memory write survived crash after {k} ops"
n_ops = remember(last)
print(f"Atomicity: crashes injected after 1..{n_ops - 1} of {n_ops} writes → memory unchanged {s0} ✔; clean commit → {mem_state()}")

# Fact supersession: one current role per (person, site); the update is atomic
def current_facts(pid):
    return rows(mem, "MATCH (a:Entity {id:$p})-[r:RELATED]->(b:Entity) RETURN r.rel, b.name, r.ts ORDER BY r.ts DESC", {"p": pid})
remember(dict(id="EP-9000", ts=1_740_000_000, session="S99", topic="personnel", txt="Faisal is the night shift supervisor on RIG-07"))
f1 = [f for f in current_facts("faisal") if f[1] == "RIG-07"]
remember(dict(id="EP-9001", ts=1_740_003_600, session="S99", topic="personnel", txt="Faisal moved to the day shift rotation on RIG-07"))
f2 = [f for f in current_facts("faisal") if f[1] == "RIG-07"]
assert len(f2) == 1 and f2[0][0] == "role:day shift", f2
print(f"Supersession: Faisal@RIG-07 {f1[0][0]} → {f2[0][0]} (exactly one current role) ✔")

# Durability: close, reopen, everything is still there (committed writes are in the WAL / checkpoint).
# Reopened read-write because LOAD is an admin statement that read-only databases refuse.
s_before = mem_state(); mem.close(); mem_db.close()
mem_db = neug.Database(mem_db_path, mode="w"); mem = mem_db.connect()
MEM_EXT_RO = load_extensions(mem)
assert mem_state() == s_before
print(f"Durability: closed and reopened {os.path.basename(mem_db_path)}, state {mem_state()} ✔")

# Hybrid recall: vector (episodic) + BM25 (keyword) + graph (entities) → integer RRF
ep_rows = rows(mem, "MATCH (e:Episode) RETURN e.id, e.txt, e.emb")
ep_ids = [r[0] for r in ep_rows]; ep_txt = {r[0]: r[1] for r in ep_rows}; ep_M = np.array([r[2] for r in ep_rows], np.float32)
bm25 = BM25(ep_txt)
def recall_lists(query, n=60):
    qv = embed_texts(mem_enc, [query])[0]
    vec = neug_vector_topk(mem, "Episode", qv, n) if MEM_EXT_RO["vector_search"] else top_k_cosine(qv, ep_ids, ep_M, n)
    kw = []
    if MEM_EXT_RO["fts"] and fts_query_text(query):
        try:
            kw = [r[0] for r in rows(mem, f"MATCH (e:Episode) RETURN e.id, bm25(e.txt, $q) AS s ORDER BY s ASC LIMIT {n}", {"q": fts_query_text(query)})]
        except Exception as ex:
            print("FTS query failed, python BM25 fallback:", str(ex)[:100]); kw = bm25.topk(query, n)
    else:
        kw = bm25.topk(query, n)
    q_ents, _ = extract(query)
    gr = []
    if q_ents:
        gr = [r[0] for r in rows(mem, "MATCH (e:Episode)-[:MENTIONS]->(n:Entity) WHERE n.id IN $ents "
                                      "RETURN e.id, count(n) AS hits, e.ts ORDER BY hits DESC, e.ts DESC LIMIT " + str(n), {"ents": q_ents})]
    return {"vector": vec, "bm25": kw, "graph": gr}
def recall(query, k=CFG["TOP_K"], use=("vector", "bm25", "graph")):
    L = recall_lists(query); lists = [L[u] for u in use]
    fused, _ = fuse(lists, set().union(*map(set, lists)) if any(lists) else set())
    return fused[:k]

# Evaluation queries with ground truth (relevant = same topic AND mentions every queried entity)
def gt(topic, ents): return {e["id"] for e in episodes if e["topic"] == topic and set(ents) <= set(e["ents"])}
QUERIES = [("Which languages does Rimal Energy want for alert messages?", "client_pref", ["rimal"]),
           ("How fast must Najm Offshore STOP-WORK alerts escalate?", "client_pref", ["najm"]),
           ("Who supervises the night shift on RIG-11?", "personnel", ["rig11"]),
           ("Who is the HSE manager for Rimal Energy?", "personnel", ["rimal"]),
           ("Why did PPE detector v3 false positives rise on RIG-07?", "model_ops", ["ppe_model", "rig07"]),
           ("Who owns retraining for the OGI camera?", "model_ops", ["ogi_cam"]),
           ("When is the H2S gateway firmware update on RIG-07?", "hardware", ["h2s_gw", "rig07"]),
           ("Did the OGI camera on RIG-11 lose its network link?", "hardware", ["ogi_cam", "rig11"]),
           ("Is the Najm Offshore invoice for RIG-07 due?", "billing", ["najm", "rig07"]),
           ("Did Rimal Energy ask for a quote for more cameras on RIG-11?", "billing", ["rimal", "rig11"]),
           ("What digest format does Najm Offshore prefer?", "client_pref", ["najm"]),
           ("Which bracket does the H2S gateway on RIG-11 need?", "hardware", ["h2s_gw", "rig11"])]
QUERIES = [q for q in QUERIES if gt(q[1], q[2])]
print(f"\nRecall evaluation: {len(QUERIES)} queries × {len(ep_ids)} episodes; each query retrieves k = |relevant| (R-precision cut); "
      f"binary MCC over all (query, episode) pairs")
RESULTS["memory_recall_mcc"] = {}
for use in [("vector",), ("bm25",), ("graph",), ("vector", "bm25"), ("vector", "bm25", "graph")]:
    yt, yp = [], []
    for q, topic, ents in QUERIES:
        rel = gt(topic, ents); got = set(recall(q, k=len(rel), use=use))
        for e in ep_ids: yt.append(e in rel); yp.append(e in got)
    cm = confusion(yt, yp); m = mcc_binary(*cm)
    RESULTS["memory_recall_mcc"]["+".join(use)] = round(m, 4)
    print(f"   {'+'.join(use):<20} MCC {m:.3f}   (tp,tn,fp,fn)={cm}")

# Answering with memory: /chat/completions grounded on recalled episodes + current facts
q = "Who supervises the night shift on RIG-11, and what does Rimal Energy want for alerts?"
hits = recall(q, k=5)
q_ents, _ = extract(q)
facts = [f"{a} {r.split(':', 1)[1]} {b}" for a, r, b in rows(
    mem, "MATCH (a:Entity)-[r:RELATED]->(b:Entity) WHERE a.id IN $e OR b.id IN $e RETURN a.name, r.rel, b.name ORDER BY r.ts DESC",
    {"e": q_ents})][:8]
ans = chat([{"role": "system", "content": "Answer using only the memory provided. Cite episode ids."},
            {"role": "user", "content": json.dumps({"question": q, "episodes": {h: ep_txt[h] for h in hits}, "facts": facts})}])
print(f"\nQ: {q}" + (f"  [{CFG['LLM_MODEL']}]" if ans else ""))
print(ans or ("Offline answer (extractive)\n  current facts: " + ("; ".join(facts) or "—") +
              "\n  episodes: " + " | ".join(f"[{h}] {ep_txt[h]}" for h in hits[:3])))

# ═══════════════════════════ Summary ═══════════════════════════
banner("Summary")
RESULTS["lean_theorems"] = THEOREMS
RESULTS["runtime_mirrors"] = ["mcc_class_swap", "mcc_all_wrong", "fuse_monotone", "fail_anywhere_rolls_back (VG360 + memory)", "within2_monotone"]
RESULTS["extensions"] = {"vg360": VG_EXT, "memory": MEM_EXT_RO}
print(json.dumps(RESULTS, indent=2, default=str))
vg.close(); vg_db.close(); mem.close(); mem_db.close()
